# ECABSD — Full Kaggle Training & Validation Pipeline

> **Goal**: Run the complete ECABSD pipeline on Kaggle GPU T4 to generate  
> publication-quality metrics with homology-aware splits and k-fold cross-validation.

### What this notebook does:
1. ✅ Install all dependencies
2. ✅ Clone ECABSD from GitHub
3. ✅ Install MMseqs2 for homology-aware splitting
4. ✅ Download DB5 benchmark structures
5. ✅ Prepare dataset (PDB → residue graphs)
6. ✅ Generate homology-aware splits (≤30% identity)
7. ✅ Train V3 model (GATv2 + Cross-Attention)
8. ✅ Run 5-fold cross-validation
9. ✅ Final evaluation — full metric report
10. ✅ Push results to GitHub

**Runtime estimate:** ~6–8 hours on Kaggle GPU T4 (within 9h limit)  
**GPU required:** Yes — enable in Notebook Settings → Accelerator → GPU T4


## ⚙️ Cell 1 — GPU & Environment Check

In [ ]:
import subprocess, sys, os

# Verify GPU
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU detected — enable GPU in Notebook Settings!')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

print(f'\nPython: {sys.version}')
print(f'Working dir: {os.getcwd()}')

## 📦 Cell 2 — Install Dependencies

> This takes ~5 minutes. Run once per Kaggle session.

In [ ]:
import subprocess, sys

def run(cmd, **kwargs):
    print(f'$ {cmd}')
    result = subprocess.run(cmd, shell=True, text=True, capture_output=True, **kwargs)
    if result.stdout: print(result.stdout[-2000:])  # last 2000 chars
    if result.returncode != 0:
        print(f'STDERR: {result.stderr[-1000:]}')
    return result.returncode

# Install ONLY base torch-geometric (takes 5 seconds, no compiling!)
print("Installing PyTorch Geometric...")
run('pip install -q torch-geometric')

# Project dependencies
print("Installing other project dependencies...")
run('pip install -q biopython pydssp transformers==4.40.2 fair-esm '    'fastapi uvicorn typer pyyaml scikit-learn tqdm matplotlib seaborn '    'python-multipart pandas')

print('\n✅ Dependencies installed!')

## 🔧 Cell 3 — Install MMseqs2

In [ ]:
import os
import requests
import tarfile
import subprocess

print("Installing MMseqs2...")

# 1. Paths
target_dir = '/tmp'
tar_path = '/tmp/mmseqs.tar.gz'
extract_dir = '/tmp/mmseqs'

# 2. Clean up any previous attempts
if os.path.exists(tar_path): os.remove(tar_path)
subprocess.run(f'rm -rf {extract_dir}', shell=True)

# 3. Download the guaranteed SSE4.1 static Linux binary
url = "https://github.com/soedinglab/MMseqs2/releases/download/15-6f452/mmseqs-linux-sse41.tar.gz"
response = requests.get(url, stream=True, allow_redirects=True, timeout=30)
with open(tar_path, 'wb') as f:
    for chunk in response.iter_content(chunk_size=8192):
        if chunk: f.write(chunk)

# 4. Extract
with tarfile.open(tar_path, "r:gz") as tar:
    tar.extractall(path=target_dir)
os.remove(tar_path)

# 5. Make executable
binary_path = os.path.join(extract_dir, 'bin/mmseqs')
os.chmod(binary_path, 0o755)

# 6. Add to system PATH permanently for this notebook session
os.environ['PATH'] = os.path.join(extract_dir, 'bin') + ':' + os.environ['PATH']

# 7. Final Verification (Checking if the program runs and outputs help text)
result = subprocess.run('mmseqs', shell=True, capture_output=True, text=True)
output_text = result.stderr if result.stderr else result.stdout

if "MMseqs2" in output_text:
    print("\n🎉 SUCCESS: MMseqs2 is fully installed and working perfectly!")
else:
    print("\n❌ Error: MMseqs2 installation failed.")

## 📁 Cell 4 — Clone ECABSD Repository

> Set your GitHub token below to also enable pushing results back at the end.

In [ ]:
import os, subprocess

# ── CONFIG — edit these ────────────────────────────────────────────────────
GITHUB_TOKEN = ''        # optional: paste your GitHub PAT to push results
GITHUB_USER  = 'amanigreeva'
GITHUB_REPO  = 'ECABSD'
WORK_DIR     = '/kaggle/working/ecabsd'
# ──────────────────────────────────────────────────────────────────────────

if os.path.exists(WORK_DIR):
    print(f'Repo already exists at {WORK_DIR} — pulling latest...')
    subprocess.run(f'git -C {WORK_DIR} pull origin main', shell=True)
else:
    if GITHUB_TOKEN:
        clone_url = f'https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{GITHUB_REPO}.git'
    else:
        clone_url = f'https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git'
    
    print(f'Cloning {GITHUB_REPO}...')
    ret = subprocess.run(f'git clone {clone_url} {WORK_DIR}', shell=True,
                         capture_output=True, text=True)
    if ret.returncode == 0:
        print(f'✅ Cloned to {WORK_DIR}')
    else:
        print(f'ERROR: {ret.stderr}')
        raise RuntimeError('Clone failed')

# Set working directory
os.chdir(WORK_DIR)
print(f'Working directory: {os.getcwd()}')

# Add to Python path
import sys
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

# Show repo structure
subprocess.run('ls -la', shell=True)

## 📥 Cell 5 — Download DB5 Benchmark Dataset

Uses the Docking Benchmark 5 (DB5) — 230 high-quality protein–protein complexes.  
Small enough to download fast, high enough quality for publication.

In [ ]:
import os, subprocess

RAW_DIR  = '/kaggle/working/data/raw/pdbs'
os.makedirs(RAW_DIR, exist_ok=True)

# Count existing PDBs
existing = [f for f in os.listdir(RAW_DIR) if f.endswith('.pdb')]
print(f'Existing PDB files: {len(existing)}')

if len(existing) < 10:
    print('Downloading DB5 benchmark structures...')
    ret = subprocess.run(
        f'python scripts/download_benchmarks.py --output-dir {RAW_DIR}',
        shell=True, capture_output=True, text=True
    )
    print(ret.stdout[-3000:] if ret.stdout else '')
    if ret.returncode != 0:
        print(f'WARN: {ret.stderr[-1000:]}')
        
        # Fallback: download individual PDBs using BioPython
        print('\nFallback: downloading PDBs via BioPython...')
        ret2 = subprocess.run(
            f'python scripts/download_pdbs.py --output-dir {RAW_DIR} --limit 250',
            shell=True, capture_output=True, text=True
        )
        print(ret2.stdout[-2000:])

existing = [f for f in os.listdir(RAW_DIR) if f.endswith('.pdb')]
print(f'\n✅ Total PDB files ready: {len(existing)}')
print('Sample:', existing[:5])

## 🔬 Cell 6 — Prepare Dataset (PDB → Residue Graphs)

In [ ]:
import os, subprocess

PROCESSED_DIR = '/kaggle/working/data/processed'
SPLITS_CSV    = '/kaggle/working/data/splits.csv'
os.makedirs(PROCESSED_DIR, exist_ok=True)

# Count existing graphs
existing_graphs = [f for f in os.listdir(PROCESSED_DIR) if f.endswith('.pt')]
print(f'Existing graph files: {len(existing_graphs)}')

if len(existing_graphs) < 50:
    print('Building residue graphs from PDB files...')
    print('(This step uses BioPython + ESM-2 embeddings — takes ~30-60 min)\n')
    
    ret = subprocess.run(
        f'python scripts/prepare_kaggle_dips.py '
        f'--pdb-dir {RAW_DIR} '
        f'--output-dir {PROCESSED_DIR} '
        f'--cutoff 4.5 '
        f'--train-ratio 0.70 '
        f'--val-ratio 0.15 '
        f'--threads 4',
        shell=True
    )
    
    if ret != 0:
        # Try the generic prepare_dataset.py
        print('Trying generic prepare_dataset.py...')
        subprocess.run(
            f'python scripts/prepare_dataset.py '
            f'--pdb-dir {RAW_DIR} '
            f'--output-dir {PROCESSED_DIR} '
            f'--cutoff 4.5 '
            f'--threads 4',
            shell=True
        )

# Verify
existing_graphs = [f for f in os.listdir(PROCESSED_DIR) if f.endswith('.pt')]
print(f'\n✅ Graph files ready: {len(existing_graphs)}')

if os.path.exists(SPLITS_CSV):
    import pandas as pd
    df = pd.read_csv(SPLITS_CSV)
    print(f'Splits CSV: {len(df)} complexes')
    print(df['split'].value_counts())
else:
    print('⚠️ splits.csv not found — check prepare step above')

## 🧬 Cell 7 — Generate Homology-Aware Splits (MMseqs2, ≤30% identity)

This is the key step for publication-grade metrics.  
Removes homologous sequences across train/val/test splits.

In [ ]:
import os, subprocess

HOMOLOGY_SPLITS = '/kaggle/working/data/splits_homology.csv'

ret = subprocess.run(
    f'python scripts/generate_homology_splits.py '
    f'--splits {SPLITS_CSV} '
    f'--pdb-dir {RAW_DIR} '
    f'--output {HOMOLOGY_SPLITS} '
    f'--identity 0.30 '
    f'--coverage 0.80 '
    f'--threads 4',
    shell=True
)

if os.path.exists(HOMOLOGY_SPLITS):
    import pandas as pd
    df_h = pd.read_csv(HOMOLOGY_SPLITS)
    print(f'\n✅ Homology-aware splits:')
    print(df_h['split'].value_counts())
    
    # Verify zero leakage
    t = set(df_h[df_h['split']=='train']['pdb_id'])
    v = set(df_h[df_h['split']=='val']['pdb_id'])
    e = set(df_h[df_h['split']=='test']['pdb_id'])
    print(f'\nLeakage check (should all be 0):')
    print(f'  Train-Val overlap:  {len(t & v)}')
    print(f'  Train-Test overlap: {len(t & e)}')
    print(f'  Val-Test overlap:   {len(v & e)}')
    
    ACTIVE_SPLITS = HOMOLOGY_SPLITS
    print(f'\n✅ Using homology-aware splits for training')
else:
    print('⚠️ Homology splits failed — falling back to random splits')
    ACTIVE_SPLITS = SPLITS_CSV

print(f'Active splits: {ACTIVE_SPLITS}')

## 🏋️ Cell 8 — Update config.yaml with Kaggle Paths

In [ ]:
import yaml, os

config_path = 'config.yaml'
with open(config_path) as f:
    cfg = yaml.safe_load(f)

# Override paths for Kaggle environment
cfg['data']['processed_dir'] = PROCESSED_DIR
cfg['data']['splits_csv']    = ACTIVE_SPLITS
cfg['paths']['checkpoints_dir'] = '/kaggle/working/checkpoints'
cfg['paths']['logs_dir']        = '/kaggle/working/logs'

# Kaggle-tuned training params
cfg['training']['epochs']   = 80    # within 9h limit
cfg['training']['num_workers'] = 2  # Kaggle has 2 CPU cores

os.makedirs('/kaggle/working/checkpoints', exist_ok=True)
os.makedirs('/kaggle/working/logs', exist_ok=True)
os.makedirs('/kaggle/working/results', exist_ok=True)

# Write updated config
kaggle_config = 'config_kaggle.yaml'
with open(kaggle_config, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)

print('✅ Kaggle config written to config_kaggle.yaml')
print(f"  processed_dir: {cfg['data']['processed_dir']}")
print(f"  splits_csv:    {cfg['data']['splits_csv']}")
print(f"  epochs:        {cfg['training']['epochs']}")
print(f"  hidden_dim:    {cfg['model']['hidden_dim']}")
print(f"  num_heads:     {cfg['model']['num_heads']}")

## 🚀 Cell 9 — Train V3 Model

**Estimated time:** 3–5 hours on GPU T4  
Best checkpoint saved to `/kaggle/working/checkpoints/best_model_v3.pt`

In [ ]:
import sys, os
sys.path.insert(0, WORK_DIR)

from train import run_training

print('🚀 Starting V3 training...')
print('Checkpoint will be saved to: /kaggle/working/checkpoints/best_model_v3.pt')
print('='*60)

run_training(config_path='config_kaggle.yaml')

## 📊 Cell 10 — Evaluate Best Checkpoint (Single Split)

In [ ]:
import torch, json, os, sys
import numpy as np
sys.path.insert(0, WORK_DIR)

from models.ecabsd_model import ECABSDModel
from data.dataset import BindingSiteDataset, collate_fn
from torch.utils.data import DataLoader
from train import validate, build_criterion, load_config
import torch.nn as nn

CKPT = '/kaggle/working/checkpoints/best_model_v3.pt'

if not os.path.exists(CKPT):
    print(f'ERROR: Checkpoint not found at {CKPT}')
    print('Please run the training cell first.')
else:
    cfg    = load_config('config_kaggle.yaml')
    mcfg   = cfg['model']
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    model = ECABSDModel(
        input_dim=mcfg.get('esm_dim', 33),
        hidden_dim=mcfg['hidden_dim'],
        num_heads=mcfg['num_heads'],
        dropout=mcfg['dropout'],
        edge_dim=mcfg.get('edge_feature_dim', 5),
        num_gcn_layers=mcfg.get('num_gcn_layers', 6),
    ).to(device)
    
    ckpt = torch.load(CKPT, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f'Loaded checkpoint from epoch {ckpt["epoch"]+1}')
    print(f'Saved val F1: {ckpt.get("best_val_f1", "N/A"):.4f}')
    print(f'Saved threshold: {ckpt.get("best_threshold", 0.5):.4f}')
    
    test_dataset = BindingSiteDataset(
        cfg['data']['processed_dir'],
        cfg['data']['splits_csv'],
        split='test'
    )
    test_loader = DataLoader(
        test_dataset, batch_size=cfg['training']['batch_size'],
        shuffle=False, num_workers=0, collate_fn=collate_fn
    )
    
    print(f'\nTest set size: {len(test_dataset)} complexes')
    
    criterion = nn.BCEWithLogitsLoss()
    test_metrics = validate(model, test_loader, criterion, device)
    
    print('\n' + '='*50)
    print('  ECABSD V3 — Test Set Results (Homology-Filtered)')
    print('='*50)
    for k, v in test_metrics.items():
        if isinstance(v, float):
            print(f'  {k:<20}: {v:.4f}')
    print('='*50)
    
    # Save metrics to file
    results_path = '/kaggle/working/results/test_metrics.json'
    with open(results_path, 'w') as f:
        json.dump(test_metrics, f, indent=2)
    print(f'\nSaved to: {results_path}')

## 🔄 Cell 11 — 5-Fold Cross-Validation

**Estimated time:** 4–6 hours (5 × shorter training runs)  
Reports mean ± std — required for peer review.

In [ ]:
import subprocess

KFOLD_OUTPUT = '/kaggle/working/results/kfold_results.json'

print('🔄 Starting 5-fold cross-validation...')
print('This runs 5 complete training cycles — expect ~4-6 hours total')
print('='*60)

ret = subprocess.run(
    f'python scripts/train_kfold.py '
    f'--config config_kaggle.yaml '
    f'--splits {ACTIVE_SPLITS} '
    f'--folds 5 '
    f'--output {KFOLD_OUTPUT} '
    f'--seed 42',
    shell=True
)

print(f'Return code: {ret}')

## 📋 Cell 12 — Print Final Results Summary

In [ ]:
import json, os

print('\n' + '='*60)
print('  ECABSD — FINAL RESULTS SUMMARY')
print('='*60)

# Single split results
test_path = '/kaggle/working/results/test_metrics.json'
if os.path.exists(test_path):
    with open(test_path) as f:
        m = json.load(f)
    print('\n📊 Test Set Results (Homology-Aware Split):')
    print(f'  F1-Score:  {m.get("f1", 0):.4f}')
    print(f'  ROC-AUC:   {m.get("auc_roc", 0):.4f}')
    print(f'  PR-AUC:    {m.get("auc_pr", 0):.4f}')
    print(f'  Recall:    {m.get("recall", 0):.4f}')
    print(f'  Precision: {m.get("precision", 0):.4f}')
    print(f'  Accuracy:  {m.get("accuracy", 0):.4f}')
    print(f'  MCC:       {m.get("mcc", 0):.4f}')

# K-fold results
kfold_path = '/kaggle/working/results/kfold_results.json'
if os.path.exists(kfold_path):
    with open(kfold_path) as f:
        kf = json.load(f)
    summary = kf.get('summary', {})
    print('\n📊 5-Fold Cross-Validation Results:')
    print(f"  {'Metric':<15} {'Mean':>8} {'±Std':>8}")
    print(f"  {'-'*35}")
    for k in ['f1', 'auc_roc', 'auc_pr', 'precision', 'recall', 'accuracy', 'mcc']:
        if k in summary:
            print(f"  {k:<15} {summary[k]['mean']:>8.4f} {summary[k]['std']:>8.4f}")

print('\n' + '='*60)
print('Copy the numbers above into RESULTS.md before pushing!')
print('='*60)

## 💾 Cell 13 — Download Checkpoint & Results

Downloads the checkpoint and results to Kaggle output — you can then download them locally.

In [ ]:
import shutil, os

OUTPUT_DIR = '/kaggle/working/output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

files_to_save = [
    ('/kaggle/working/checkpoints/best_model_v3.pt', 'best_model_v3.pt'),
    ('/kaggle/working/results/test_metrics.json',    'test_metrics.json'),
    ('/kaggle/working/results/kfold_results.json',   'kfold_results.json'),
    ('/kaggle/working/results/kfold_results.txt',    'kfold_results.txt'),
    (ACTIVE_SPLITS,                                  'splits_used.csv'),
]

print('Files saved to /kaggle/working/output/ (downloadable from Kaggle UI):')
for src, dst_name in files_to_save:
    dst = os.path.join(OUTPUT_DIR, dst_name)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        size = os.path.getsize(dst) / 1e6
        print(f'  ✅ {dst_name} ({size:.1f} MB)')
    else:
        print(f'  ⚠️  {src} not found')

## 🚀 Cell 14 — Push Results Back to GitHub (Optional)

Only runs if you set `GITHUB_TOKEN` in Cell 4.  
Updates `RESULTS.md` and pushes the new checkpoint.

In [ ]:
import json, os, subprocess

if not GITHUB_TOKEN:
    print('Skipping push — GITHUB_TOKEN not set.')
    print('Download results from /kaggle/working/output/ instead.')
else:
    os.chdir(WORK_DIR)

    # Configure git
    subprocess.run(f'git config user.email "{GITHUB_USER}@users.noreply.github.com"', shell=True)
    subprocess.run(f'git config user.name "{GITHUB_USER}"', shell=True)

    # Copy results into repo
    import shutil
    os.makedirs('results', exist_ok=True)
    os.makedirs('checkpoints', exist_ok=True)

    for src, dst in [
        ('/kaggle/working/results/test_metrics.json',  'results/test_metrics_homology.json'),
        ('/kaggle/working/results/kfold_results.json', 'results/kfold_results.json'),
        ('/kaggle/working/results/kfold_results.txt',  'results/kfold_results.txt'),
    ]:
        if os.path.exists(src):
            shutil.copy2(src, dst)
            print(f'Copied {src} → {dst}')

    # Update RESULTS.md with actual numbers
    test_path  = 'results/test_metrics_homology.json'
    kfold_path = 'results/kfold_results.json'

    update_lines = []
    if os.path.exists(test_path):
        with open(test_path) as f:
            m = json.load(f)
        update_lines += [
            f"\n## Homology-Filtered Results (MMseqs2 ≤30% identity)\n",
            f"| Metric | Score |\n|---|---|\n",
            f"| **F1 Score** | `{m.get('f1',0):.4f}` |\n",
            f"| **ROC-AUC** | `{m.get('auc_roc',0):.4f}` |\n",
            f"| **PR-AUC** | `{m.get('auc_pr',0):.4f}` |\n",
            f"| **Recall** | `{m.get('recall',0):.4f}` |\n",
            f"| **Precision** | `{m.get('precision',0):.4f}` |\n",
            f"| **Accuracy** | `{m.get('accuracy',0):.4f}` |\n",
            f"| **MCC** | `{m.get('mcc',0):.4f}` |\n",
        ]

    if os.path.exists(kfold_path):
        with open(kfold_path) as f:
            kf = json.load(f)
        summary = kf.get('summary', {})
        update_lines += [
            f"\n## 5-Fold Cross-Validation Results\n",
            f"| Metric | Mean | ±Std |\n|---|---|---|\n",
        ]
        for k in ['f1', 'auc_roc', 'auc_pr', 'mcc']:
            if k in summary:
                update_lines.append(
                    f"| **{k.upper()}** | `{summary[k]['mean']:.4f}` | `{summary[k]['std']:.4f}` |\n"
                )

    if update_lines:
        with open('RESULTS.md', 'a') as f:
            f.write('\n---\n')
            f.writelines(update_lines)
        print('Updated RESULTS.md with new metrics')

    # Commit and push
    subprocess.run('git add results/ RESULTS.md', shell=True)
    subprocess.run('git commit -m "results: add homology-filtered and k-fold CV metrics from Kaggle run"', shell=True)
    subprocess.run('git push origin main', shell=True)
    print('\n✅ Pushed to GitHub!')

## ✅ Done!

### What to do next:

1. **Download from Kaggle UI**: Go to the output tab and download `/kaggle/working/output/`
2. **Copy checkpoint** `best_model_v3.pt` into your local `checkpoints/` folder
3. **Update RESULTS.md** with the real numbers from Cell 12
4. **Push to GitHub**: `git add -A && git commit -m 'results: homology-filtered k-fold metrics' && git push`

### After that, the repo is fully publication-ready:
| Item | Status |
|---|---|
| Architecture (GATv2 V3) | ✅ |
| Homology-aware splits | ✅ |
| K-fold CV metrics | ✅ |
| Checkpoint (reproducible) | ✅ |
| RESULTS.md with methodology | ✅ |
| CI (GitHub Actions) | ✅ |
| CITATION.cff, environment.yml | ✅ |
